<a href="https://colab.research.google.com/github/mohasemps/Public-Projects/blob/master/GNGO_DT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Gulu NGO Donation Tracker – FINAL STABLE (with debug & upsert)

import subprocess
import sys
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

import contextlib
with contextlib.redirect_stdout(None), contextlib.redirect_stderr(None):
    subprocess.run([sys.executable, "-m", "pip", "install", "flask", "flask-cors",
                    "sqlalchemy", "python-jose", "pydantic", "werkzeug", "-q"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import uuid
import hashlib
import secrets
from datetime import datetime, timedelta
from functools import wraps
import threading
import time
import logging
logging.getLogger('werkzeug').disabled = True
logging.getLogger('sqlalchemy').setLevel(logging.CRITICAL)

from flask import Flask, request, jsonify, render_template_string, redirect, url_for, make_response
from flask_cors import CORS
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, Text, ForeignKey, Boolean, func
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, relationship
from jose import jwt


# Database Setup

DATABASE_URL = "sqlite:///./donations.db?check_same_thread=False"
SECRET_KEY = "donation-tracker-secret-key-2026"
ALGORITHM = "HS256"
TOKEN_EXPIRE_DAYS = 7

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False}, echo=False)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# Password utilities (unchanged)
def hash_password(password: str) -> str:
    salt = secrets.token_hex(16)
    hash_obj = hashlib.pbkdf2_hmac('sha256', password.encode(), salt.encode(), 100000)
    return f"{salt}${hash_obj.hex()}"

def verify_password(plain: str, hashed: str) -> bool:
    try:
        salt, hash_val = hashed.split('$')
        hash_obj = hashlib.pbkdf2_hmac('sha256', plain.encode(), salt.encode(), 100000)
        return hash_obj.hex() == hash_val
    except:
        return False

def create_token(data: dict) -> str:
    data['exp'] = datetime.utcnow() + timedelta(days=TOKEN_EXPIRE_DAYS)
    return jwt.encode(data, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        username = payload.get("sub")
        role = payload.get("role")
        if username:
            return username, role
        return None, None
    except Exception as e:
        print(f"⚠️ Token verification error: {e}")  # visible in console
        return None, None


# Models

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    username = Column(String, unique=True)
    password = Column(String)
    role = Column(String, default="viewer")

class Donor(Base):
    __tablename__ = "donors"
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    email = Column(String, unique=True)
    phone = Column(String)
    address = Column(String)
    donor_type = Column(String, default="individual")
    is_active = Column(Boolean, default=True)
    created_at = Column(DateTime, default=datetime.utcnow)
    donations = relationship("Donation", back_populates="donor")

class Project(Base):
    __tablename__ = "projects"
    id = Column(Integer, primary_key=True)
    name = Column(String, nullable=False)
    code = Column(String, unique=True)
    description = Column(Text)
    goal_amount = Column(Float, default=0.0)
    raised_amount = Column(Float, default=0.0)
    start_date = Column(DateTime, default=datetime.utcnow)
    end_date = Column(DateTime)
    is_active = Column(Boolean, default=True)
    donations = relationship("Donation", back_populates="project")

class Donation(Base):
    __tablename__ = "donations"
    id = Column(Integer, primary_key=True)
    donor_id = Column(Integer, ForeignKey("donors.id"), nullable=False)
    project_id = Column(Integer, ForeignKey("projects.id"))
    amount = Column(Float, nullable=False)
    currency = Column(String, default="UGX")
    donation_type = Column(String, default="cash")
    description = Column(Text)
    date = Column(DateTime, default=datetime.utcnow)
    receipt_number = Column(String, unique=True)
    is_anonymous = Column(Boolean, default=False)
    donor = relationship("Donor", back_populates="donations")
    project = relationship("Project", back_populates="donations")

Base.metadata.create_all(bind=engine)


# Core Functions

def generate_receipt_number():
    return f"DON-{uuid.uuid4().hex[:6].upper()}"

def register_donor(db, name, email, phone=None, address=None, donor_type="individual"):
    if db.query(Donor).filter(Donor.email == email).first():
        raise ValueError(f"Donor with email {email} already exists.")
    donor = Donor(name=name, email=email, phone=phone, address=address, donor_type=donor_type)
    db.add(donor)
    db.commit()
    db.refresh(donor)
    return donor

def get_donor(db, donor_id):
    return db.query(Donor).filter(Donor.id == donor_id).first()

def list_donors(db, active_only=True):
    q = db.query(Donor)
    if active_only:
        q = q.filter(Donor.is_active == True)
    return q.all()

def update_donor(db, donor_id, **kwargs):
    donor = get_donor(db, donor_id)
    if not donor:
        raise ValueError(f"Donor {donor_id} not found.")
    for key, value in kwargs.items():
        if hasattr(donor, key):
            setattr(donor, key, value)
    db.commit()
    db.refresh(donor)
    return donor

def create_project(db, name, description="", goal_amount=0.0, end_date=None):
    code = f"PRJ-{uuid.uuid4().hex[:6].upper()}"
    project = Project(name=name, code=code, description=description,
                      goal_amount=goal_amount, end_date=end_date)
    db.add(project)
    db.commit()
    db.refresh(project)
    return project

def get_project(db, project_id):
    return db.query(Project).filter(Project.id == project_id).first()

def list_projects(db, active_only=True):
    q = db.query(Project)
    if active_only:
        q = q.filter(Project.is_active == True)
    return q.all()

def update_project_funding(db, project_id):
    project = get_project(db, project_id)
    if not project:
        raise ValueError(f"Project {project_id} not found.")
    total = db.query(func.sum(Donation.amount)).filter(Donation.project_id == project_id).scalar() or 0.0
    project.raised_amount = total
    db.commit()
    db.refresh(project)
    return project

def record_donation(db, donor_id, amount, project_id=None, currency="UGX",
                    donation_type="cash", description="", is_anonymous=False):
    donor = get_donor(db, donor_id)
    if not donor:
        raise ValueError(f"Donor {donor_id} not found.")
    if project_id:
        project = get_project(db, project_id)
        if not project:
            raise ValueError(f"Project {project_id} not found.")
    receipt = generate_receipt_number()
    donation = Donation(
        donor_id=donor_id,
        project_id=project_id,
        amount=amount,
        currency=currency,
        donation_type=donation_type,
        description=description,
        receipt_number=receipt,
        is_anonymous=is_anonymous
    )
    db.add(donation)
    db.commit()
    db.refresh(donation)
    if project_id:
        update_project_funding(db, project_id)
    return donation

def get_donation(db, donation_id):
    return db.query(Donation).filter(Donation.id == donation_id).first()

def list_donations(db, donor_id=None, project_id=None, start_date=None, end_date=None, limit=None):
    q = db.query(Donation)
    if donor_id:
        q = q.filter(Donation.donor_id == donor_id)
    if project_id:
        q = q.filter(Donation.project_id == project_id)
    if start_date:
        q = q.filter(Donation.date >= start_date)
    if end_date:
        q = q.filter(Donation.date <= end_date)
    if limit:
        q = q.limit(limit)
    return q.all()

def donation_summary_by_donor(db, donor_id):
    donor = get_donor(db, donor_id)
    if not donor:
        raise ValueError(f"Donor {donor_id} not found.")
    total = db.query(func.sum(Donation.amount)).filter(Donation.donor_id == donor_id).scalar() or 0.0
    count = db.query(Donation).filter(Donation.donor_id == donor_id).count()
    return {"donor": donor.name, "total_amount": total, "donation_count": count}

def project_donation_report(db, project_id):
    project = get_project(db, project_id)
    if not project:
        raise ValueError(f"Project {project_id} not found.")
    donations = list_donations(db, project_id=project_id)
    total = project.raised_amount
    return {
        "project": project.name,
        "code": project.code,
        "goal": project.goal_amount,
        "raised": total,
        "remaining": project.goal_amount - total,
        "donations": [
            {
                "date": d.date.isoformat(),
                "amount": d.amount,
                "donor": d.donor.name if not d.is_anonymous else "Anonymous",
                "type": d.donation_type,
                "receipt": d.receipt_number
            }
            for d in donations
        ]
    }

def overall_summary(db, start_date=None, end_date=None):
    q = db.query(Donation)
    if start_date:
        q = q.filter(Donation.date >= start_date)
    if end_date:
        q = q.filter(Donation.date <= end_date)
    total_amount = q.with_entities(func.sum(Donation.amount)).scalar() or 0.0
    total_donations = q.count()
    unique_donors = db.query(Donor).join(Donation).distinct().count()
    active_projects = db.query(Project).filter(Project.is_active == True).count()
    return {
        "total_donations": total_donations,
        "total_amount": total_amount,
        "unique_donors": unique_donors,
        "active_projects": active_projects,
        "period": f"{start_date or 'start'} to {end_date or 'now'}"
    }


# User management (with upsert for seed)

def create_user(db, username, password, role="viewer"):
    if db.query(User).filter(User.username == username).first():
        raise ValueError(f"User {username} already exists.")
    user = User(username=username, password=hash_password(password), role=role)
    db.add(user)
    db.commit()
    db.refresh(user)
    return user

def get_user_by_username(db, username):
    return db.query(User).filter(User.username == username).first()

def list_users(db):
    return db.query(User).all()

def update_user_role(db, user_id, new_role):
    user = db.query(User).filter(User.id == user_id).first()
    if not user:
        raise ValueError(f"User {user_id} not found.")
    user.role = new_role
    db.commit()
    db.refresh(user)
    return user


# Flask App

app = Flask(__name__)
app.secret_key = SECRET_KEY
CORS(app, supports_credentials=True)

import werkzeug.serving
werkzeug.serving.WSGIRequestHandler.log = lambda self, type, message, *args: None
werkzeug.serving.WSGIRequestHandler.log_request = lambda *args, **kwargs: None

def require_auth(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        token = request.cookies.get('auth_token')
        if not token:
            auth_header = request.headers.get('Authorization', '')
            token = auth_header.replace('Bearer ', '')
        if not token:
            if request.path.startswith('/api/'):
                return jsonify({"error": "Missing authentication token"}), 401
            return redirect(url_for('login_page'))

        username, role = verify_token(token)
        if not username:
            print(f"❌ Invalid token for request: {request.path}")  # console debug
            if request.path.startswith('/api/'):
                return jsonify({"error": "Invalid or expired token"}), 401
            # Clear invalid cookie and redirect
            resp = make_response(redirect(url_for('login_page')))
            resp.set_cookie('auth_token', '', expires=0, path='/')
            return resp

        request.user = {"username": username, "role": role}
        return f(*args, **kwargs)
    return decorated

def require_role(allowed_roles):
    def decorator(f):
        @wraps(f)
        def decorated(*args, **kwargs):
            if not hasattr(request, 'user') or not request.user:
                token = request.cookies.get('auth_token') or request.headers.get('Authorization', '').replace('Bearer ', '')
                if not token:
                    return jsonify({"error": "Unauthorized"}), 401
                username, role = verify_token(token)
                if not username:
                    return jsonify({"error": "Invalid token"}), 401
                request.user = {"username": username, "role": role}
            if request.user['role'] not in allowed_roles:
                return jsonify({"error": "Insufficient permissions"}), 403
            return f(*args, **kwargs)
        return decorated
    return decorator


# Login Page

LOGIN_HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Gulu NGO Donation Tracker - Login</title>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            min-height: 100vh;
            background: linear-gradient(135deg, #0a3a3a 0%, #1a6b6b 50%, #0a3a3a 100%);
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .login-container { width: 100%; max-width: 420px; margin: 20px; }
        .login-card {
            background: rgba(255,255,255,0.98);
            border-radius: 20px;
            padding: 40px 35px;
            box-shadow: 0 20px 40px rgba(0,0,0,0.3);
        }
        .badge { text-align: center; margin-bottom: 20px; }
        .badge span {
            font-size: 60px;
            background: #1a6b6b;
            display: inline-block;
            width: 80px;
            height: 80px;
            line-height: 80px;
            border-radius: 50%;
            color: white;
        }
        h2 { text-align: center; color: #1a6b6b; font-size: 26px; }
        .subtitle { text-align: center; color: #5a6e8a; font-size: 13px; margin-bottom: 30px; border-bottom: 1px solid #e0e7ff; padding-bottom: 15px; }
        .input-group { margin-bottom: 20px; }
        .input-group label { display: block; font-size: 14px; font-weight: 600; color: #2c3e66; margin-bottom: 6px; }
        .input-group input {
            width: 100%;
            padding: 14px 16px;
            border: 1px solid #ccd7f0;
            border-radius: 12px;
            font-size: 15px;
            background: #f9fbfe;
        }
        .input-group input:focus {
            outline: none;
            border-color: #1a6b6b;
            box-shadow: 0 0 0 3px rgba(26,107,107,0.2);
        }
        button {
            width: 100%;
            padding: 14px;
            background: linear-gradient(90deg, #1a6b6b, #2a8a8a);
            color: white;
            border: none;
            border-radius: 12px;
            font-size: 16px;
            font-weight: bold;
            cursor: pointer;
            transition: all 0.2s;
            margin-top: 10px;
        }
        button:hover { transform: scale(1.01); box-shadow: 0 4px 12px rgba(26,107,107,0.3); }
        .error { color: #dc3545; background: #ffe6e6; padding: 10px; border-radius: 10px; text-align: center; margin-top: 20px; }
        .footer { text-align: center; margin-top: 25px; font-size: 11px; color: #6c7a8e; }
    </style>
</head>
<body>
    <div class="login-container">
        <div class="login-card">
            <div class="badge"><span>❤️</span></div>
            <h2>Gulu NGO Donation Tracker</h2>
            <div class="subtitle">Empowering Communities through Transparency</div>
            <form method="post">
                <div class="input-group">
                    <label>👤 Username</label>
                    <input type="text" name="username" placeholder="Enter username" required autofocus>
                </div>
                <div class="input-group">
                    <label>🔒 Password</label>
                    <input type="password" name="password" placeholder="Enter password" required>
                </div>
                <button type="submit">Login →</button>
                {% if error %}
                <div class="error">{{ error }}</div>
                {% endif %}
            </form>
            <div class="footer"><i>Authorized personnel only • Gulu NGO</i></div>
        </div>
    </div>
</body>
</html>
"""

@app.route('/login', methods=['GET', 'POST'])
def login_page():
    if request.method == 'POST':
        username = request.form.get('username')
        password = request.form.get('password')
        db = SessionLocal()
        try:
            user = get_user_by_username(db, username)
            if user and verify_password(password, user.password):
                token = create_token({"sub": user.username, "role": user.role})
                resp = make_response(redirect(url_for('index')))
                resp.set_cookie('auth_token', token, httponly=True, secure=False, samesite='Lax', path='/')
                print(f"✅ Login successful: {username} (role: {user.role})")
                return resp
            else:
                print(f"❌ Login failed for: {username}")
                return render_template_string(LOGIN_HTML, error="Invalid username or password")
        finally:
            db.close()
    return render_template_string(LOGIN_HTML, error=None)

@app.route('/logout')
def logout():
    resp = make_response(redirect(url_for('login_page')))
    resp.set_cookie('auth_token', '', expires=0, path='/')
    return resp


# Dashboard HTML

DASHBOARD_HTML = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Gulu NGO Donation Tracker</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #f0f7f7 0%, #dce8e8 100%);
            min-height: 100vh;
            padding: 20px;
        }
        .container { max-width: 1400px; margin: 0 auto; }
        .header {
            background: white;
            border-radius: 20px;
            padding: 20px 30px;
            margin-bottom: 25px;
            display: flex;
            justify-content: space-between;
            align-items: center;
            flex-wrap: wrap;
            box-shadow: 0 10px 25px rgba(0,0,0,0.1);
            border-left: 6px solid #1a6b6b;
        }
        .logo { display: flex; align-items: center; gap: 15px; }
        .logo .badge {
            background: #1a6b6b;
            width: 50px;
            height: 50px;
            border-radius: 50%;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 28px;
            color: white;
        }
        .logo h1 { color: #1a6b6b; font-size: 26px; }
        .logo p { color: #5a6e8a; font-size: 12px; }
        .user-info { font-size: 14px; color: #1a6b6b; }
        .btn {
            padding: 10px 24px;
            border: none;
            border-radius: 40px;
            cursor: pointer;
            font-size: 14px;
            font-weight: bold;
            transition: all 0.3s;
        }
        .btn-primary { background: #1a6b6b; color: white; }
        .btn-primary:hover { background: #2a8a8a; transform: translateY(-2px); }
        .btn-danger { background: #dc3545; color: white; }
        .btn-danger:hover { background: #c82333; transform: translateY(-2px); }
        .btn-success { background: #28a745; color: white; }
        .btn-success:hover { background: #218838; transform: translateY(-2px); }
        .btn-warning { background: #ffc107; color: #333; }
        .btn-sm { padding: 6px 12px; font-size: 12px; }
        .stats {
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 20px;
            margin-bottom: 25px;
        }
        .stat-card {
            background: white;
            border-radius: 20px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 5px 15px rgba(0,0,0,0.1);
            border-bottom: 3px solid #1a6b6b;
        }
        .stat-card h3 { font-size: 36px; color: #1a6b6b; margin-bottom: 10px; }
        .tabs {
            display: flex;
            flex-wrap: wrap;
            gap: 12px;
            margin-bottom: 25px;
        }
        .tab {
            padding: 10px 20px;
            background: white;
            border: none;
            border-radius: 40px;
            cursor: pointer;
            font-size: 14px;
            font-weight: 600;
            transition: all 0.2s;
            color: #1a6b6b;
            box-shadow: 0 2px 8px rgba(0,0,0,0.05);
        }
        .tab.active {
            background: #1a6b6b;
            color: white;
            box-shadow: 0 4px 10px rgba(0,0,0,0.2);
        }
        .tab:hover:not(.active) { background: #e0f0f0; }
        .tab.hidden { display: none; }
        .card {
            background: white;
            border-radius: 20px;
            padding: 25px;
            box-shadow: 0 8px 20px rgba(0,0,0,0.1);
        }
        .card h2 {
            margin-bottom: 20px;
            color: #1a6b6b;
            border-left: 4px solid #1a6b6b;
            padding-left: 15px;
        }
        input, select, textarea {
            width: 100%;
            padding: 12px;
            margin: 8px 0;
            border: 1px solid #ccd7f0;
            border-radius: 12px;
            background: white;
        }
        input:focus, select:focus, textarea:focus {
            outline: none;
            border-color: #1a6b6b;
            box-shadow: 0 0 0 3px rgba(26,107,107,0.2);
        }
        .grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 20px; }
        @media (max-width: 768px) { .grid { grid-template-columns: 1fr; } .stats { grid-template-columns: 1fr 1fr; } }
        .list-item {
            background: #f8f9fa;
            padding: 15px;
            margin-bottom: 12px;
            border-radius: 15px;
            border-left: 4px solid #1a6b6b;
        }
        .badge {
            display: inline-block;
            padding: 4px 10px;
            border-radius: 20px;
            font-size: 11px;
            font-weight: bold;
        }
        .badge-success { background: #28a745; color: white; }
        .badge-warning { background: #ffc107; color: #333; }
        .badge-danger { background: #dc3545; color: white; }
        .badge-primary { background: #1a6b6b; color: white; }
        .badge-info { background: #17a2b8; color: white; }
        table { width: 100%; border-collapse: collapse; }
        th, td { padding: 12px; text-align: left; border-bottom: 1px solid #e0e7ff; }
        th { background: #1a6b6b; color: white; border-radius: 12px 12px 0 0; }
        .footer {
            margin-top: 30px;
            text-align: center;
            color: #5a6e8a;
            font-size: 12px;
            padding: 15px;
            border-top: 1px solid rgba(0,0,0,0.1);
        }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="logo">
                <div class="badge">❤️</div>
                <div>
                    <h1>Gulu NGO Donation Tracker</h1>
                    <p>Transparent Aid Management</p>
                </div>
            </div>
            <div>
                <span class="user-info" id="userRoleDisplay"></span>
                <a href="/logout" class="btn btn-danger">🚪 Logout</a>
            </div>
        </div>

        <div class="stats">
            <div class="stat-card"><h3 id="statDonations">0</h3><p>💰 Total Donations</p></div>
            <div class="stat-card"><h3 id="statAmount">0</h3><p>🇺🇬 Total Amount (UGX)</p></div>
            <div class="stat-card"><h3 id="statDonors">0</h3><p>👤 Active Donors</p></div>
            <div class="stat-card"><h3 id="statProjects">0</h3><p>📋 Active Projects</p></div>
        </div>

        <div class="tabs">
            <button class="tab active" onclick="showTab('dashboard')">📊 Dashboard</button>
            <button class="tab" onclick="showTab('donors')">👤 Donors</button>
            <button class="tab" onclick="showTab('projects')">📋 Projects</button>
            <button class="tab" onclick="showTab('donations')">💰 Donations</button>
            <button class="tab" onclick="showTab('reports')">📈 Reports</button>
            <button class="tab" id="usersTabBtn" onclick="showTab('users')" style="display:none;">👥 Users</button>
        </div>

        <!-- Dashboard Tab -->
        <div id="dashboardTab" class="tab-content active">
            <div class="card">
                <h2>📊 Recent Activity</h2>
                <div id="recentDonations" style="max-height:400px; overflow-y:auto;">Loading...</div>
            </div>
        </div>

        <!-- Donors Tab -->
        <div id="donorsTab" class="tab-content" style="display:none;">
            <div class="grid">
                <div class="card" id="donorFormCard">
                    <h2>➕ Register Donor</h2>
                    <form id="donorForm">
                        <input type="text" id="donorName" placeholder="Full Name" required>
                        <input type="email" id="donorEmail" placeholder="Email" required>
                        <input type="text" id="donorPhone" placeholder="Phone">
                        <input type="text" id="donorAddress" placeholder="Address">
                        <select id="donorType">
                            <option value="individual">Individual</option>
                            <option value="corporate">Corporate</option>
                            <option value="foundation">Foundation</option>
                        </select>
                        <button type="submit" class="btn btn-primary" style="width:100%">Register Donor</button>
                    </form>
                </div>
                <div class="card">
                    <h2>👤 Donor List</h2>
                    <div id="donorList" style="max-height:400px; overflow-y:auto;">Loading...</div>
                </div>
            </div>
        </div>

        <!-- Projects Tab -->
        <div id="projectsTab" class="tab-content" style="display:none;">
            <div class="grid">
                <div class="card" id="projectFormCard">
                    <h2>➕ Create Project</h2>
                    <form id="projectForm">
                        <input type="text" id="projectName" placeholder="Project Name" required>
                        <textarea id="projectDesc" rows="3" placeholder="Description"></textarea>
                        <input type="number" id="projectGoal" placeholder="Goal Amount (UGX)" required>
                        <input type="datetime-local" id="projectEnd" placeholder="End Date (optional)">
                        <button type="submit" class="btn btn-success" style="width:100%">Create Project</button>
                    </form>
                </div>
                <div class="card">
                    <h2>📋 Active Projects</h2>
                    <div id="projectList" style="max-height:400px; overflow-y:auto;">Loading...</div>
                </div>
            </div>
        </div>

        <!-- Donations Tab -->
        <div id="donationsTab" class="tab-content" style="display:none;">
            <div class="grid">
                <div class="card" id="donationFormCard">
                    <h2>💰 Record Donation</h2>
                    <form id="donationForm">
                        <select id="donorSelect" required><option value="">Select Donor</option></select>
                        <select id="projectSelect"><option value="">Select Project (optional)</option></select>
                        <input type="number" id="donationAmount" placeholder="Amount" required>
                        <select id="donationCurrency">
                            <option value="UGX">UGX</option>
                            <option value="USD">USD</option>
                            <option value="EUR">EUR</option>
                        </select>
                        <select id="donationType">
                            <option value="cash">Cash</option>
                            <option value="in_kind">In‑Kind</option>
                            <option value="service">Service</option>
                        </select>
                        <textarea id="donationDesc" rows="3" placeholder="Description"></textarea>
                        <label><input type="checkbox" id="donationAnonymous"> Anonymous</label>
                        <button type="submit" class="btn btn-primary" style="width:100%">Record Donation</button>
                    </form>
                </div>
                <div class="card">
                    <h2>📜 Donation Log</h2>
                    <div id="donationList" style="max-height:400px; overflow-y:auto;">Loading...</div>
                </div>
            </div>
        </div>

        <!-- Reports Tab -->
        <div id="reportsTab" class="tab-content" style="display:none;">
            <div class="card">
                <h2>📈 Overall Summary</h2>
                <div id="overallSummary">Loading...</div>
                <hr>
                <h2>📊 Project Report</h2>
                <div style="display:flex; gap:10px; flex-wrap:wrap;">
                    <select id="reportProjectSelect"><option value="">Select Project</option></select>
                    <button class="btn btn-primary" onclick="loadProjectReport()">Generate Report</button>
                </div>
                <div id="projectReport" style="margin-top:20px;"></div>
            </div>
        </div>

        <!-- Users Tab (Admin only) -->
        <div id="usersTab" class="tab-content" style="display:none;">
            <div class="grid">
                <div class="card">
                    <h2>👥 Create New User</h2>
                    <form id="userForm">
                        <input type="text" id="newUsername" placeholder="Username" required>
                        <input type="password" id="newPassword" placeholder="Password" required>
                        <select id="newUserRole">
                            <option value="admin">Admin</option>
                            <option value="finance_officer">Finance Officer</option>
                            <option value="viewer">Viewer</option>
                        </select>
                        <button type="submit" class="btn btn-primary" style="width:100%">Create User</button>
                    </form>
                </div>
                <div class="card">
                    <h2>📋 Existing Users</h2>
                    <div id="userList" style="max-height:400px; overflow-y:auto;">Loading...</div>
                </div>
            </div>
        </div>

        <div class="footer">© 2026 Gulu NGO Donation Tracker | Empowering Communities</div>
    </div>

    <script>
        let currentUserRole = '';

        async function fetchMe() {
            try {
                const res = await fetch('/api/me', { credentials: 'include' });
                if (!res.ok) throw new Error('Not authenticated');
                const data = await res.json();
                currentUserRole = data.role;
                document.getElementById('userRoleDisplay').textContent = '👤 ' + data.username + ' (' + data.role + ')';
                const canWrite = (currentUserRole === 'admin' || currentUserRole === 'finance_officer');
                const isAdmin = (currentUserRole === 'admin');
                document.getElementById('donorFormCard').style.display = canWrite ? 'block' : 'none';
                document.getElementById('projectFormCard').style.display = canWrite ? 'block' : 'none';
                document.getElementById('donationFormCard').style.display = canWrite ? 'block' : 'none';
                document.getElementById('usersTabBtn').style.display = isAdmin ? 'inline-block' : 'none';
                if (!isAdmin && document.getElementById('usersTab').style.display !== 'none') {
                    showTab('dashboard');
                }
            } catch(e) {
                console.error('Failed to fetch user info', e);
                window.location.href = '/login';
            }
        }

        function showTab(tab) {
            const tabs = ['dashboard', 'donors', 'projects', 'donations', 'reports', 'users'];
            const idx = tabs.indexOf(tab);
            document.querySelectorAll('.tab').forEach((t, i) => i === idx ? t.classList.add('active') : t.classList.remove('active'));
            document.querySelectorAll('.tab-content').forEach(c => c.style.display = 'none');
            document.getElementById(`${tab}Tab`).style.display = 'block';
            if (tab === 'dashboard') loadDashboard();
            if (tab === 'donors') loadDonors();
            if (tab === 'projects') loadProjects();
            if (tab === 'donations') { loadDonorsForSelect(); loadProjectsForSelect(); loadDonations(); }
            if (tab === 'reports') { loadOverallSummary(); loadProjectsForReportSelect(); }
            if (tab === 'users') { loadUsers(); }
        }

        async function fetchApi(url, options = {}) {
            const res = await fetch(url, { ...options, credentials: 'include' });
            if (!res.ok) throw new Error(await res.text());
            return res.json();
        }

        async function loadDashboard() {
            try {
                const stats = await fetchApi('/api/stats');
                document.getElementById('statDonations').textContent = stats.total_donations;
                document.getElementById('statAmount').textContent = stats.total_amount.toLocaleString();
                document.getElementById('statDonors').textContent = stats.unique_donors;
                document.getElementById('statProjects').textContent = stats.active_projects;
                const recent = await fetchApi('/api/donations/recent?limit=10');
                const container = document.getElementById('recentDonations');
                if (!recent.length) { container.innerHTML = '<p>No donations yet.</p>'; return; }
                container.innerHTML = recent.map(d => `<div class="list-item"><strong>${d.donor}</strong> donated ${d.amount.toLocaleString()} ${d.currency} to <strong>${d.project || 'General'}</strong> (${d.receipt})<br><small>${new Date(d.date).toLocaleString()}</small></div>`).join('');
            } catch(e) { console.error(e); }
        }

        async function loadDonors() {
            try {
                const donors = await fetchApi('/api/donors');
                const container = document.getElementById('donorList');
                if (!donors.length) { container.innerHTML = '<p>No donors registered.</p>'; return; }
                container.innerHTML = donors.map(d => `<div class="list-item"><strong>${d.name}</strong> (${d.donor_type})<br>📧 ${d.email} | 📞 ${d.phone || 'N/A'}<br><span class="badge badge-primary">${d.is_active ? 'Active' : 'Inactive'}</span></div>`).join('');
            } catch(e) { console.error(e); }
        }

        document.getElementById('donorForm')?.addEventListener('submit', async (e) => {
            e.preventDefault();
            const data = {
                name: document.getElementById('donorName').value,
                email: document.getElementById('donorEmail').value,
                phone: document.getElementById('donorPhone').value,
                address: document.getElementById('donorAddress').value,
                donor_type: document.getElementById('donorType').value
            };
            try {
                await fetchApi('/api/donors', { method: 'POST', headers: {'Content-Type':'application/json'}, body: JSON.stringify(data) });
                alert('Donor registered successfully!');
                document.getElementById('donorForm').reset();
                loadDonors();
            } catch(e) { alert('Error: ' + e.message); }
        });

        async function loadProjects() {
            try {
                const projects = await fetchApi('/api/projects');
                const container = document.getElementById('projectList');
                if (!projects.length) { container.innerHTML = '<p>No projects.</p>'; return; }
                container.innerHTML = projects.map(p => `<div class="list-item"><strong>${p.name}</strong> (${p.code})<br>Goal: ${p.goal_amount.toLocaleString()} UGX | Raised: ${p.raised_amount.toLocaleString()} UGX (${Math.round((p.raised_amount/p.goal_amount)*100)}%)<br>${p.is_active ? '🟢 Active' : '🔴 Closed'}</div>`).join('');
            } catch(e) { console.error(e); }
        }

        document.getElementById('projectForm')?.addEventListener('submit', async (e) => {
            e.preventDefault();
            const data = {
                name: document.getElementById('projectName').value,
                description: document.getElementById('projectDesc').value,
                goal_amount: parseFloat(document.getElementById('projectGoal').value),
                end_date: document.getElementById('projectEnd').value || null
            };
            try {
                await fetchApi('/api/projects', { method: 'POST', headers: {'Content-Type':'application/json'}, body: JSON.stringify(data) });
                alert('Project created!');
                document.getElementById('projectForm').reset();
                loadProjects();
            } catch(e) { alert('Error: ' + e.message); }
        });

        async function loadDonorsForSelect() {
            const donors = await fetchApi('/api/donors');
            const sel = document.getElementById('donorSelect');
            sel.innerHTML = '<option value="">Select Donor</option>' + donors.map(d => `<option value="${d.id}">${d.name}</option>`).join('');
        }
        async function loadProjectsForSelect() {
            const projects = await fetchApi('/api/projects');
            const sel = document.getElementById('projectSelect');
            sel.innerHTML = '<option value="">Select Project (optional)</option>' + projects.map(p => `<option value="${p.id}">${p.name}</option>`).join('');
        }
        async function loadDonations() {
            try {
                const donations = await fetchApi('/api/donations');
                const container = document.getElementById('donationList');
                if (!donations.length) { container.innerHTML = '<p>No donations recorded.</p>'; return; }
                container.innerHTML = donations.map(d => `<div class="list-item"><strong>${d.donor}</strong> donated ${d.amount.toLocaleString()} ${d.currency} to <strong>${d.project || 'General'}</strong> (${d.receipt})<br><small>${new Date(d.date).toLocaleString()} | ${d.type}</small></div>`).join('');
            } catch(e) { console.error(e); }
        }

        document.getElementById('donationForm')?.addEventListener('submit', async (e) => {
            e.preventDefault();
            const data = {
                donor_id: parseInt(document.getElementById('donorSelect').value),
                amount: parseFloat(document.getElementById('donationAmount').value),
                project_id: document.getElementById('projectSelect').value ? parseInt(document.getElementById('projectSelect').value) : null,
                currency: document.getElementById('donationCurrency').value,
                donation_type: document.getElementById('donationType').value,
                description: document.getElementById('donationDesc').value,
                is_anonymous: document.getElementById('donationAnonymous').checked
            };
            if (!data.donor_id) { alert('Please select a donor.'); return; }
            try {
                const result = await fetchApi('/api/donations', { method: 'POST', headers: {'Content-Type':'application/json'}, body: JSON.stringify(data) });
                alert(`Donation recorded! Receipt: ${result.receipt}`);
                document.getElementById('donationForm').reset();
                loadDonations();
                loadDashboard();
            } catch(e) { alert('Error: ' + e.message); }
        });

        async function loadOverallSummary() {
            try {
                const summary = await fetchApi('/api/reports/overall');
                document.getElementById('overallSummary').innerHTML = `
                    <p><strong>Total Donations:</strong> ${summary.total_donations}</p>
                    <p><strong>Total Amount:</strong> ${summary.total_amount.toLocaleString()} UGX</p>
                    <p><strong>Unique Donors:</strong> ${summary.unique_donors}</p>
                    <p><strong>Active Projects:</strong> ${summary.active_projects}</p>
                    <p><strong>Period:</strong> ${summary.period}</p>
                `;
            } catch(e) { console.error(e); }
        }

        async function loadProjectsForReportSelect() {
            const projects = await fetchApi('/api/projects');
            const sel = document.getElementById('reportProjectSelect');
            sel.innerHTML = '<option value="">Select Project</option>' + projects.map(p => `<option value="${p.id}">${p.name}</option>`).join('');
        }

        window.loadProjectReport = async function() {
            const projectId = document.getElementById('reportProjectSelect').value;
            if (!projectId) { alert('Select a project.'); return; }
            try {
                const report = await fetchApi(`/api/reports/project/${projectId}`);
                const container = document.getElementById('projectReport');
                let html = `<h3>${report.project} (${report.code})</h3>
                    <p>Goal: ${report.goal.toLocaleString()} UGX | Raised: ${report.raised.toLocaleString()} UGX | Remaining: ${report.remaining.toLocaleString()} UGX</p>
                    <table><tr><th>Date</th><th>Donor</th><th>Amount</th><th>Type</th><th>Receipt</th></tr>`;
                report.donations.forEach(d => {
                    html += `<tr><td>${new Date(d.date).toLocaleDateString()}</td><td>${d.donor}</td><td>${d.amount.toLocaleString()}</td><td>${d.type}</td><td>${d.receipt}</td></tr>`;
                });
                html += '</table>';
                container.innerHTML = html;
            } catch(e) { alert('Error: ' + e.message); }
        };

        async function loadUsers() {
            try {
                const users = await fetchApi('/api/users');
                const container = document.getElementById('userList');
                if (!users.length) { container.innerHTML = '<p>No users.</p>'; return; }
                container.innerHTML = users.map(u => `
                    <div class="list-item">
                        <strong>${u.username}</strong>
                        <span class="badge ${u.role === 'admin' ? 'badge-danger' : u.role === 'finance_officer' ? 'badge-warning' : 'badge-info'}">${u.role}</span>
                        <br>
                        <button class="btn btn-sm btn-primary" onclick="updateUserRole(${u.id}, 'admin')">Set Admin</button>
                        <button class="btn btn-sm btn-warning" onclick="updateUserRole(${u.id}, 'finance_officer')">Set Finance</button>
                        <button class="btn btn-sm btn-secondary" onclick="updateUserRole(${u.id}, 'viewer')">Set Viewer</button>
                    </div>
                `).join('');
            } catch(e) { console.error(e); }
        }

        window.updateUserRole = async function(userId, newRole) {
            if (!confirm(`Change role to ${newRole}?`)) return;
            try {
                await fetchApi(`/api/users/${userId}/role`, {
                    method: 'PUT',
                    headers: {'Content-Type':'application/json'},
                    body: JSON.stringify({ role: newRole })
                });
                alert('Role updated!');
                loadUsers();
            } catch(e) { alert('Error: ' + e.message); }
        };

        document.getElementById('userForm')?.addEventListener('submit', async (e) => {
            e.preventDefault();
            const data = {
                username: document.getElementById('newUsername').value,
                password: document.getElementById('newPassword').value,
                role: document.getElementById('newUserRole').value
            };
            try {
                await fetchApi('/api/users', {
                    method: 'POST',
                    headers: {'Content-Type':'application/json'},
                    body: JSON.stringify(data)
                });
                alert('User created!');
                document.getElementById('userForm').reset();
                loadUsers();
            } catch(e) { alert('Error: ' + e.message); }
        });

        fetchMe().then(() => {
            loadDashboard();
        });
    </script>
</body>
</html>
"""

# MAIN INDEX
@app.route('/')
@require_auth
def index():
    return render_template_string(DASHBOARD_HTML)


# API Routes

@app.route('/api/me')
@require_auth
def api_me():
    return jsonify({"username": request.user['username'], "role": request.user['role']})

@app.route('/api/stats')
@require_auth
def api_stats():
    db = SessionLocal()
    try:
        return jsonify(overall_summary(db))
    finally:
        db.close()

@app.route('/api/donors', methods=['GET'])
@require_auth
def api_list_donors():
    db = SessionLocal()
    try:
        donors = list_donors(db)
        return jsonify([{
            "id": d.id, "name": d.name, "email": d.email,
            "phone": d.phone, "address": d.address,
            "donor_type": d.donor_type, "is_active": d.is_active
        } for d in donors])
    finally:
        db.close()

@app.route('/api/donors', methods=['POST'])
@require_auth
@require_role(['admin', 'finance_officer'])
def api_register_donor():
    data = request.json
    db = SessionLocal()
    try:
        donor = register_donor(db, data['name'], data['email'],
                               data.get('phone'), data.get('address'),
                               data.get('donor_type', 'individual'))
        return jsonify({"id": donor.id, "message": "Donor registered"})
    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    finally:
        db.close()

@app.route('/api/projects', methods=['GET'])
@require_auth
def api_list_projects():
    db = SessionLocal()
    try:
        projects = list_projects(db)
        return jsonify([{
            "id": p.id, "name": p.name, "code": p.code,
            "description": p.description, "goal_amount": p.goal_amount,
            "raised_amount": p.raised_amount, "is_active": p.is_active
        } for p in projects])
    finally:
        db.close()

@app.route('/api/projects', methods=['POST'])
@require_auth
@require_role(['admin', 'finance_officer'])
def api_create_project():
    data = request.json
    db = SessionLocal()
    try:
        end_date = datetime.fromisoformat(data['end_date']) if data.get('end_date') else None
        project = create_project(db, data['name'], data.get('description', ''),
                                 data['goal_amount'], end_date)
        return jsonify({"id": project.id, "code": project.code})
    except Exception as e:
        return jsonify({"error": str(e)}), 400
    finally:
        db.close()

@app.route('/api/donations', methods=['GET'])
@require_auth
def api_list_donations():
    db = SessionLocal()
    try:
        donations = list_donations(db)
        return jsonify([{
            "id": d.id,
            "donor": d.donor.name if not d.is_anonymous else "Anonymous",
            "amount": d.amount,
            "currency": d.currency,
            "project": d.project.name if d.project else None,
            "type": d.donation_type,
            "date": d.date.isoformat(),
            "receipt": d.receipt_number
        } for d in donations])
    finally:
        db.close()

@app.route('/api/donations/recent', methods=['GET'])
@require_auth
def api_recent_donations():
    limit = request.args.get('limit', 10, type=int)
    db = SessionLocal()
    try:
        donations = list_donations(db, limit=limit)
        return jsonify([{
            "donor": d.donor.name if not d.is_anonymous else "Anonymous",
            "amount": d.amount,
            "currency": d.currency,
            "project": d.project.name if d.project else "General",
            "date": d.date.isoformat(),
            "receipt": d.receipt_number
        } for d in donations])
    finally:
        db.close()

@app.route('/api/donations', methods=['POST'])
@require_auth
@require_role(['admin', 'finance_officer'])
def api_record_donation():
    data = request.json
    db = SessionLocal()
    try:
        donation = record_donation(
            db,
            data['donor_id'],
            data['amount'],
            data.get('project_id'),
            data.get('currency', 'UGX'),
            data.get('donation_type', 'cash'),
            data.get('description', ''),
            data.get('is_anonymous', False)
        )
        return jsonify({"receipt": donation.receipt_number, "amount": donation.amount})
    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    finally:
        db.close()

@app.route('/api/reports/overall')
@require_auth
def api_overall_report():
    db = SessionLocal()
    try:
        return jsonify(overall_summary(db))
    finally:
        db.close()

@app.route('/api/reports/project/<int:project_id>')
@require_auth
def api_project_report(project_id):
    db = SessionLocal()
    try:
        return jsonify(project_donation_report(db, project_id))
    except ValueError as e:
        return jsonify({"error": str(e)}), 404
    finally:
        db.close()

@app.route('/api/users', methods=['GET'])
@require_auth
@require_role(['admin'])
def api_list_users():
    db = SessionLocal()
    try:
        users = list_users(db)
        return jsonify([{"id": u.id, "username": u.username, "role": u.role} for u in users])
    finally:
        db.close()

@app.route('/api/users', methods=['POST'])
@require_auth
@require_role(['admin'])
def api_create_user():
    data = request.json
    db = SessionLocal()
    try:
        user = create_user(db, data['username'], data['password'], data.get('role', 'viewer'))
        return jsonify({"id": user.id, "username": user.username, "role": user.role})
    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    finally:
        db.close()

@app.route('/api/users/<int:user_id>/role', methods=['PUT'])
@require_auth
@require_role(['admin'])
def api_update_user_role(user_id):
    data = request.json
    db = SessionLocal()
    try:
        user = update_user_role(db, user_id, data['role'])
        return jsonify({"id": user.id, "username": user.username, "role": user.role})
    except ValueError as e:
        return jsonify({"error": str(e)}), 404
    finally:
        db.close()


# Seed Database (UPSERT)

def seed():
    db = SessionLocal()
    try:
        # Always ensure the three users exist, using upsert logic
        users_to_create = [
            ("Admin", "Nebraska@2014", "admin"),
            ("Finance", "Nebraska@2014", "finance_officer"),
            ("Viewer", "Nebraska@2014", "viewer")
        ]
        for username, password, role in users_to_create:
            existing = get_user_by_username(db, username)
            if existing:
                # If user exists but role is wrong, update it
                if existing.role != role:
                    existing.role = role
                    db.commit()
                    print(f"🔄 Updated role for {username} to {role}")
            else:
                create_user(db, username, password, role)
                print(f"✅ Created user: {username} (role: {role})")

        # Check if donors/projects already exist to avoid duplicates
        if not db.query(Donor).first():
            donor1 = register_donor(db, "Sarah Akello", "sarah@example.com", phone="+256700000001")
            donor2 = register_donor(db, "Gulu Business Association", "gba@example.com", phone="+256700000002", donor_type="corporate")
            proj1 = create_project(db, "School Feeding Program", "Provide meals to 500 children", goal_amount=5000000)
            proj2 = create_project(db, "Clean Water Initiative", "Drill boreholes in 3 villages", goal_amount=3000000)
            record_donation(db, donor1.id, 500000, proj1.id, description="First installment")
            record_donation(db, donor2.id, 1000000, proj1.id, description="Corporate sponsorship")
            record_donation(db, donor1.id, 200000, proj2.id, description="Support water project")
            print("✅ Sample donors, projects, and donations created.")
        else:
            print("ℹ️ Sample data already exists, skipping.")
        db.commit()
    except Exception as e:
        print(f"⚠️ Seed error: {e}")
    finally:
        db.close()


# Start Server

def get_colab_url(port=8000):
    try:
        from google.colab import output
        return output.eval_js(f"google.colab.kernel.proxyPort({port})")
    except:
        return None

from IPython.display import clear_output, HTML, display
clear_output(wait=True)

print("="*60)
print(" Gulu NGO Donation Tracker – FINAL STABLE ")
print("="*60)
print("\nInitializing database...")
seed()

def run_flask():
    app.run(host='0.0.0.0', port=8000, debug=False, use_reloader=False, threaded=True)

print("\nStarting Flask server on port 8000...")
server_thread = threading.Thread(target=run_flask, daemon=True)
server_thread.start()
time.sleep(3)

proxy_url = get_colab_url(8000)
if proxy_url:
    link_html = f"""
    <div style="background-color: #f0f7ff; padding: 20px; border-radius: 10px; border-left: 4px solid #1a6b6b; margin: 20px 0;">
        <p style="font-size: 18px; margin: 0 0 10px 0; color: #1a6b6b;"><b>Click below to open the Donation Tracker:</b></p>
        <a href="{proxy_url}" target="_blank" style="font-size: 16px; color: white; text-decoration: none; background-color: #1a6b6b; padding: 10px 20px; border-radius: 5px; display: inline-block;">
            🚀 Open Gulu Donation Tracker
        </a>
    </div>
    """
    display(HTML(link_html))
else:
    print("\n🌐 Server running at http://localhost:8000")

print("\n🔑 Login Credentials (case‑sensitive):")
print("   Admin:     Admin / Nebraska@2014")
print("   Finance:   Finance / Nebraska@2014")
print("   Viewer:    Viewer / Nebraska@2014")
print("\n⚠️ Keep this cell running. Press Ctrl+C to stop.\n")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Server stopped.")